# Claims Investigation Agent — Eval-Driven Development

**Workshop flow**
1. Setup: install dependencies, set API keys, clone repo
2. Build & run the agent: see the trace in LangSmith
3. Spot the problem: agent hallucinates in its rationale
4. Apply the grounding evaluator: catch the failure systematically
5. Fix the prompt: one line change
6. Re-run & compare: grounding score improves

## 1 · Setup

In [ ]:
import sys, os

# Add repo root to sys.path so 'data' and 'evals' are importable
repo_root = os.path.abspath(os.path.join(os.path.dirname("__file__"), ".."))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

assert os.path.exists(os.path.join(repo_root, "data/sources/policy_docs.md")), \
    f"Expected to find data/sources/policy_docs.md under {repo_root}"
print("Repo root:", repo_root)

# Run once if packages are missing. Skip if you installed via: uv sync
# !pip install -q langchain-openai langgraph langsmith openevals python-dotenv

In [ ]:
# Reads OPENAI_API_KEY and LANGSMITH_API_KEY from .env at the repo root.
from dotenv import load_dotenv
load_dotenv()

import os
os.environ.setdefault('LANGSMITH_TRACING', 'true')
os.environ.setdefault('LANGSMITH_PROJECT', 'claims-workshop')
print('OPENAI_API_KEY set:   ', bool(os.getenv('OPENAI_API_KEY')))
print('LANGSMITH_API_KEY set:', bool(os.getenv('LANGSMITH_API_KEY')))

## 2 · Build the agent

The agent reads an incoming claim *(inc. claimant, incident_location, reported_cause, estimated_amount_eur)* and decides which tools to call. 

The agent has four tools: one per data source. At inference time it decides which ones to call based on the claim and the policy.

<img src="assets/agent_diagram.png" width="600"/>

The **policy document is the source of truth**, its clauses define exactly which checks are required and under what conditions.

| Clause | What it covers | Check triggered |
|--------|---------------|-----------------|
| **1** — Water damage | Burst pipes / valve failures, if reported within **72h** and cost > **€5k** | baseline coverage rule |
| **2** — Weather structural | Storm damage if wind > **90 km/h** or rain > **40mm/24h** | → `query_weather_data` when cause is weather-related |
| **3** — Contractor work | Damage within **30 days** of licensed contractor work | → `query_claims_history` to check prior contractor activity |
| **4** — Negligence exclusion | Known defects left unrepaired > **90 days** are excluded | → `query_claims_history` to check for documented prior defects |
| **5** — Repeat claims | 20% excess applied on 2nd+ same-type claim within **36 months** | → always call `query_claims_history` |
| **6** — Repair estimate | Itemised contractor estimate required for claims > **€10k** | → `retrieve_repair_estimate` when amount exceeds threshold |

In [ ]:
from langchain_core.tools import tool
from data.loaders import load_policy_docs, load_claims_history, load_weather_data, load_repair_estimate

@tool
def search_policy_docs() -> str:
    """Retrieve insurance policy clauses, coverage conditions, thresholds, and exclusions.
    Always call this first."""
    return load_policy_docs()

@tool
def query_claims_history(claimant_id: str) -> list:
    """Retrieve prior claims history for a claimant ID.
    Always call this to check for repeat claims."""
    return load_claims_history(claimant_id)

@tool
def query_weather_data(incident_date: str, location: str) -> dict:
    """Retrieve historical weather for an incident date (YYYY-MM-DD) and city.
    Call when the cause could be weather-related. Clause 2 thresholds: 40mm or 90 km/h."""
    return load_weather_data(incident_date, location)

@tool
def retrieve_repair_estimate(claim_id: str) -> dict:
    """Retrieve contractor repair estimate. Required by Clause 6 for claims over €10,000."""
    return load_repair_estimate(claim_id)

tools = [search_policy_docs, query_claims_history, query_weather_data, retrieve_repair_estimate]
print("Tools:", [t.name for t in tools])

In [ ]:
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from IPython.display import Image

PROMPT_BEFORE = """You are a claims investigation assistant for an insurance group.

You have access to four tools:
- search_policy_docs: retrieves the insurance policy
- query_claims_history: retrieves prior claims for a claimant
- query_weather_data: retrieves historical weather for a date and location
- retrieve_repair_estimate: retrieves contractor repair estimate for a claim

Investigate the claim and provide:
- coverage_decision (covered / partial / excluded)
- settlement_recommendation (auto_settle / assign_adjuster / flag_for_investigation)
- confidence (high / medium / low)
- rationale explaining your decision"""

llm          = ChatOpenAI(model="gpt-4o", temperature=0)
agent_before = create_agent(llm, tools, system_prompt=PROMPT_BEFORE)
print("Agent ready")
Image(agent_before.get_graph().draw_mermaid_png())

> **Note:** We're using LangChain's `create_agent` here, which lets the LLM decide which sources to check based on the claim context.
> 
> This keeps the architecture flexible as you add new data sources: each new source is a single `@tool` definition, no routing logic to update. The tradeoff is *non-deterministic tool selection*, which is exactly the failure mode we'll catch with the grounding evaluator in Section 4.
> 
> If you want to enforce deterministic tool selection, you can use an explicit **LangGraph graph** that enforces which checks are always required, making your investigation procedures consistent across every claim, but which requires more maintenance if the logic needs to be updated.

## 3 · Run the agent

Let's test the agent on 2 claims. We'll collect both the agent response and the outputs of the tools he called, accessible from the messages returned by the agent:

In [ ]:
# Let's check what a claim look like
from evals.dataset import CLAIM_INPUTS
CLAIM_INPUTS[0]

In [ ]:
def run_and_get_tool_outputs(claim: dict, agent):
    """Run the agent once and return (retrieved_content, response)."""
    # create_agent speaks in messages, not structured objects, so we need to parse the structured input as a string
    claim_text = "\n".join(f"{k}: {v}" for k, v in claim.items())
    agent_output = agent.invoke({"messages": [HumanMessage(content=claim_text)]})
    tool_outputs = {}
    for msg in agent_output["messages"]:
        if getattr(msg, "type", None) == "tool":
            tool_outputs[msg.name] = str(msg.content)
    retrieved = "\n\n".join(f"### {k}\n{v}" for k, v in tool_outputs.items())
    response  = agent_output["messages"][-1].content
    return retrieved, response

In [ ]:
# Claim 1 - clean baseline (CLM003, Utrecht, 9800 EUR, no prior history)
from evals.dataset import CLAIM_INPUTS
claim1 = CLAIM_INPUTS[3]  # CLAIM-2022-0441

print("Input:", claim1["claim_id"], "|", claim1["estimated_amount_eur"], "|", claim1["reported_cause"])
print()
retrieved_tools_claim1, response_claim1 = run_and_get_tool_outputs(claim1, agent_before)
print("=== Agent response ===")
print(response_claim1)

In [ ]:
# Claim 2 - fraud signals (CLM005, Amsterdam, 9400 EUR, weather data contradicts reported cause)
claim2 = CLAIM_INPUTS[0]  # CLAIM-2024-0312

print("Input:", claim2["claim_id"], "|", claim2["estimated_amount_eur"], "|", claim2["reported_cause"])
print()
retrieved_tools_claim2, response_claim2 = run_and_get_tool_outputs(claim2, agent_before)
print("=== Agent response ===")
print(response_claim2)

> The recommendations look reasonable, but nothing in the output tells us whether the rationale is based on retrieved evidence or the model's prior knowledge.
> 
> A €28,900 claim approved on memory rather than verified data is a liability.
Let's build a grounding evaluator to catch this systematically.

## 4 · Apply the grounding evaluator

The agent produced a recommendation, but can we trust the rationale?

**Is every claim in the rationale supported by what the tools actually returned?**

We'll ask an LLM judge to compare the agent's response against the outputs of the tools retrieved, which we'll collect from the agent state messages.

<img src="assets/grounding_eval_diagram.png" width="600"/>

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

GROUNDING_PROMPT = """You are evaluating whether an AI agent's response is grounded
in the evidence it actually retrieved from its tools.

## Tool outputs retrieved during investigation:
{retrieved_content}

## Agent's final response:
{response}

Is the response grounded in the tool outputs above?
- GROUNDED: the main factual claims and coverage decision trace to 
  specific retrieved content. Minor hedging on ambiguous policy 
  interpretations is acceptable if the core reasoning is supported.
- PARTIALLY_GROUNDED: a significant factual claim in the coverage 
  decision or settlement recommendation is not supported by retrieved content.
- HALLUCINATED: the response asserts facts clearly absent from 
  the retrieved content.

Reply with one of: GROUNDED, PARTIALLY_GROUNDED, HALLUCINATED
Then one sentence explaining which claim is unsupported (if any)."""

# Using a LLM judge to reason over the content and detect whether a specific claim is supported, even when paraphrased
# gpt-4o-mini handles comparison tasks reliably and is cheaper than other models
_judge_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def call_judge(retrieved_content: str, response: str) -> dict:
    prompt = GROUNDING_PROMPT.format(retrieved_content=retrieved_content, response=response)
    result = _judge_llm.invoke([HumanMessage(content=prompt)])
    text = result.content.strip()
    first_line = text.split(":")[0].upper()
    comment = text.split(":", 1)[1].strip() if ":" in text else ""
    return {"score": first_line, "comment": comment}

print("Judge ready")

> **Note 1:** This evaluator checks whether the rationale is consistent with what the tools returned, not whether the tools returned the right thing. If the data sources themselves contain errors, the agent will look grounded even if its conclusion is wrong. In production, add a separate data quality layer to validate the source data before it reaches the agent.

> **Note 2:** Libraries like openevals and agentevals provide prebuilt versions of this pattern. We build from scratch so you understand what is inside the black box before deciding whether to use one.

In [ ]:
def _score(verdict: str) -> float:
    if "GROUNDED" in verdict and "PARTIAL" not in verdict and "HALL" not in verdict:
        return 1.0
    return 0.5 if "PARTIAL" in verdict else 0.0


def show_result(result: dict, response: str) -> None:
    score = _score(result["score"])
    label = {1.0: "✅ GROUNDED", 0.5: "⚠️  PARTIAL", 0.0: "❌ HALLUCINATED"}[score]
    print(f"Score:   {label}")
    print(f"Comment: {result['comment']}")
    print()
    print("--- Response ---")
    print(response)

In [ ]:
print("Input:", claim1["claim_id"], "|", claim1["estimated_amount_eur"], "|", claim1["reported_cause"])
print()
print("--- Groundedness Assessment of Claim #1 ---")
claim1_judged_before = call_judge(retrieved_tools_claim1, response_claim1)
show_result(claim1_judged_before, response_claim1)

In [ ]:
print("Input:", claim2["claim_id"], "|", claim2["estimated_amount_eur"], "|", claim2["reported_cause"])
print()
print("--- Groundedness Assessment of Claim #2 ---")
claim2_judged_before = call_judge(retrieved_tools_claim2, response_claim2)
show_result(claim2_judged_before, response_claim2)

## 5 · Fix the prompt & re-evaluate

The evaluator flagged an ungrounded response. Now we update the prompt (same tools, same agent, one new instruction) and measure whether the grounding score improves.

In [ ]:
PROMPT_AFTER = """You are a claims investigation assistant for EuroShield Insurance Group.

You have access to four tools:
- search_policy_docs: retrieves the insurance policy
- query_claims_history: retrieves prior claims for a claimant
- query_weather_data: retrieves historical weather for a date and location
- retrieve_repair_estimate: retrieves contractor repair estimate for a claim

Always start by reading the policy to understand which checks are required.
Let the policy clauses guide which other tools you call.

Investigate the claim and provide:
- coverage_decision (covered / partial / excluded)
- settlement_recommendation (auto_settle / assign_adjuster / flag_for_investigation)
- confidence (high / medium / low)
- rationale explaining your decision

Rules for your rationale:
1. Quote exact text from tool outputs to support each claim.
2. If tool outputs contain contradictory findings, explicitly acknowledge
   the contradiction and reflect it in your coverage decision.
3. Never assert a coverage decision that goes beyond what the retrieved
   evidence confirms. If evidence is inconclusive, your coverage decision
   must be 'partial' or 'excluded', not 'covered'.
4. If the policy does not address a specific scenario, quote the relevant
   clause verbatim and state that the scenario is not explicitly covered
   by the policy text — do not infer an interpretation."""

agent_after = create_agent(llm, tools, system_prompt=PROMPT_AFTER)

print("Input:", claim2["claim_id"], "|", claim2["estimated_amount_eur"], "|", claim2["reported_cause"])
print()
retrieved_tools_after, response_after = run_and_get_tool_outputs(claim2, agent_after)
print("--- Groundedness Assessment of Claim #2 ---")
claim2_judged_after = call_judge(retrieved_tools_after, response_after)
show_result(claim2_judged_after, response_after)

> **Note:** LLM outputs are non-deterministic even at temperature=0.
> The grounding score may vary between runs, which is itself an important
> observation. In production you'd run each claim multiple times and
> aggregate scores to get a stable signal. A claim that oscillates between
> GROUNDED and PARTIAL is itself a finding: the agent's reasoning on that
> claim is inconsistent and warrants closer inspection.

## 6 · Run across all claims with LangSmith evaluate()

Scale the same evaluator across the full dataset and compare experiments in LangSmith.

The pattern has three parts:

- `target()` runs the agent on each example in the dataset and returns the message state
- `grounding_evaluator()` agent run as a Python object from `evaluate()`, extracts tool outputs and the final response from its message history, then calls the judge
- `evaluate()` from LangSmith orchestrates both, runs them in parallel, and pushes results to the dataset

To compare before and after:
1. Set `active_agent = agent_before` and `prompt_version = "before"`, run the cell
2. Set `active_agent = agent_after` and `prompt_version = "after"`, run the cell again
3. Open LangSmith, go to the dataset, and click "Compare experiments" to see the score difference side by side

In [ ]:
from langsmith import Client
from langsmith.evaluation import evaluate

DATASET_NAME = "claims-investigation-workshop"
client = Client()

# Push dataset once, safe to re-run, skips if already exists
if DATASET_NAME not in {d.name for d in client.list_datasets()}:
    ds = client.create_dataset(dataset_name=DATASET_NAME)
    for claim in CLAIM_INPUTS:
        client.create_example(inputs={"claim": claim}, dataset_id=ds.id)
    print(f"Created dataset with {len(CLAIM_INPUTS)} examples.")
else:
    print(f"Dataset '{DATASET_NAME}' already exists.")

In [ ]:
# swap active_agent and prompt_version to run the other experiment
active_agent   = agent_after   # or agent_before
prompt_version = "after"       # or "before"

def target(inputs: dict) -> dict:
    claim_text = "\n".join(f"{k}: {v}" for k, v in inputs["claim"].items())
    state = active_agent.invoke({"messages": [HumanMessage(content=claim_text)]})
    return {"messages": state["messages"]}

def grounding_evaluator(run, example):
    tool_outputs = {}
    for msg in run.outputs.get("messages", []):
        if getattr(msg, "type", None) == "tool":
            tool_outputs[msg.name] = str(msg.content)
    retrieved = "\n\n".join(f"### {k}\n{v}" for k, v in tool_outputs.items()) or "No tool outputs."
    response  = run.outputs["messages"][-1].content
    result    = call_judge(retrieved, response)
    score     = _score(result["score"])
    return {"key": "grounding", "score": score, "comment": result["comment"]}

results = evaluate(
    target,
    data=DATASET_NAME,
    evaluators=[grounding_evaluator],
    experiment_prefix="grounding",
    metadata={"prompt_version": prompt_version},
)

scores = [r["evaluation_results"]["results"][0].score for r in results]
labels = {1.0: "✅ GROUNDED", 0.5: "⚠️  PARTIAL", 0.0: "❌ HALLUCINATED"}
for claim, score in zip(CLAIM_INPUTS, scores):
    print(f"  {claim['claim_id']}  ->  {labels.get(score, score)}")
print(f"\nMean grounding score: {sum(scores)/len(scores):.2f}")

print("\nView experiment comparison in LangSmith:")
print("https://smith.langchain.com/")
print(f"Open project 'claims-workshop' -> Datasets -> {DATASET_NAME} -> Compare experiments")